# necrohub: QLoRA fine-tune for report-to-JSON extraction

Fine-tunes a small open instruct model on the synthetic necrohub corpus so it turns a redacted pt-BR
necropsy report into the extraction JSON, then compares it with the base model on the test split,
exports it to GGUF and prepares the Ollama model. Everything is regenerated from the seed.

- Decisions: staged fine-tune (DECISIONS row 6), base model, training stack and export route
  (rows 22 and 23 in `docs/DECISIONS.md`).
- Base model: `Qwen/Qwen3-4B-Instruct-2507` (Apache-2.0, ungated, non-thinking). Fallback:
  `Qwen/Qwen3-1.7B` (Apache-2.0). Portuguese quality is not stated on the 2507 card: this notebook
  measures it (base vs fine-tuned, same test split).
- Runtime: a free cloud T4 (Colab or Kaggle). The run measures its own wall times and writes them to
  `results/qlora_extraction.json`.
- The data is 100 % synthetic. The model is a demonstration: not for clinical or forensic use.
- Weights are never published to the Hugging Face Hub from this notebook.

Sources: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507, https://huggingface.co/Qwen/Qwen3-1.7B,
https://huggingface.co/docs/trl/sft_trainer, https://docs.ollama.com/import,
https://docs.ollama.com/capabilities/structured-outputs.

In [ ]:
NECROHUB_REF = "main"  # pin to a commit sha for a reproducible run
REPO_URL = "https://github.com/emanuel-luis/necrohub.git"
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
FALLBACK_MODEL = "Qwen/Qwen3-1.7B"  # set BASE_MODEL to it if the 4B model does not fit
N_CASES = 1500
SEED = 7
MAX_EVAL = None  # cap on test examples scored per model; None scores the whole split
MAX_NEW_TOKENS = 768
EVAL_BATCH_SIZE = 8
LLAMA_CPP_TAG = "b8996"
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LEARNING_RATE = 2e-4
EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 8
MAX_LENGTH = 2048
WORK = "/content/work"
CKPT = "/content/ckpt"

In [ ]:
import subprocess

import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)
assert torch.cuda.is_available(), "no GPU: switch the runtime to a T4"
print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

## Data side

The corpus is bit-identical only with the locked faker, fpdf2, pypdf, ftfy and pydantic versions, and
the repo needs Python 3.12 while the notebook kernel may not be 3.12. So every necrohub command runs
in a uv-managed environment synced from `uv.lock` (core dependencies only, no torch). The kernel
itself never imports necrohub; it reads the exported JSONL.

In [ ]:
import json
import os
import shlex
import time
from pathlib import Path

import yaml

T0 = time.time()
WALL = {}
os.makedirs(WORK, exist_ok=True)


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")


def necrohub(*args):
    cmd = ["uv", "run", "--locked", "--directory", "necrohub", "necrohub", *args]
    sh(" ".join(shlex.quote(a) for a in cmd))


sh("pip install -q uv")
if not Path("necrohub").exists():
    sh(f"git clone {REPO_URL} necrohub")
sh(f"git -C necrohub checkout {shlex.quote(NECROHUB_REF)}")
sh("git -C necrohub log -1 --format=%H")

In [ ]:
params = yaml.safe_load(Path("necrohub/params.yaml").read_text("utf-8"))
params["generate"]["n_cases"] = N_CASES
params["seed"] = SEED
PARAMS = f"{WORK}/params.yaml"
Path(PARAMS).write_text(yaml.safe_dump(params, sort_keys=False), encoding="utf-8")
common = ["--params", PARAMS, "--runs-dir", f"{WORK}/runs", "--metrics-dir", f"{WORK}/metrics"]
t = time.time()
necrohub("generate", *common, "--out", f"{WORK}/raw")
necrohub("ingest", *common, "--in", f"{WORK}/raw", "--out", f"{WORK}/interim")
necrohub(
    "preprocess",
    *common,
    "--in",
    f"{WORK}/interim",
    "--out",
    f"{WORK}/processed",
    "--raw",
    f"{WORK}/raw",
)
necrohub(
    "build-dataset",
    *common,
    "--processed",
    f"{WORK}/processed",
    "--raw",
    f"{WORK}/raw",
    "--out",
    f"{WORK}/datasets",
)
necrohub("export-sft", *common, "--data", f"{WORK}/datasets", "--out", f"{WORK}/sft")
WALL["data_s"] = round(time.time() - t, 1)

built = json.loads(Path(f"{WORK}/metrics/build_dataset.json").read_text("utf-8"))
reference = json.loads(Path("necrohub/docs/results/full-corpus.json").read_text("utf-8"))
assert built["datasets_sha256"] == reference["datasets_sha256"], "the corpus is not the reference"
DATASET_SHA = built["datasets_sha256"]
print("datasets sha256 matches docs/results/full-corpus.json:", DATASET_SHA)

## Model side

Pinned installs into the kernel (Colab's preinstalled CUDA torch is kept). Restart the runtime if pip
asks for it, then run the cells below again from the top of this section.

In [ ]:
sh(
    "pip install -q transformers==5.20.2 trl==1.16.0 peft==0.21.2 bitsandbytes==0.50.2 "
    "accelerate==1.15.0 datasets"
)

In [ ]:
import gc

import bitsandbytes
import peft
import transformers
import trl
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

VERSIONS = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "peft": peft.__version__,
    "bitsandbytes": bitsandbytes.__version__,
}
print(VERSIONS)


def read_rows(split):
    path = Path(f"{WORK}/sft/{split}.jsonl")
    return [json.loads(x) for x in path.read_text("utf-8").splitlines() if x.strip()]


train_rows, val_rows, test_rows = (read_rows(s) for s in ("train", "validation", "test"))
test_extraction = [r for r in test_rows if r["task"] == "extraction"]
if MAX_EVAL:
    test_extraction = test_extraction[:MAX_EVAL]
print(len(train_rows), len(val_rows), len(test_extraction))

# ChatML with generation markers, so assistant_only_loss trains on the answers only. It renders
# the same text as Qwen's template for system/user/assistant turns and as the Modelfile's.
CHAT_TEMPLATE = (
    "{% for m in messages %}"
    "{% if m['role'] == 'assistant' %}"
    "{{ '<|im_start|>assistant\n' }}{% generation %}{{ m['content'] + '<|im_end|>\n' }}"
    "{% endgeneration %}"
    "{% else %}{{ '<|im_start|>' + m['role'] + '\n' + m['content'] + '<|im_end|>\n' }}{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.chat_template = CHAT_TEMPLATE
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def load_4bit(adapter=None):
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=BNB, dtype=torch.float16, device_map={"": 0}
    )
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    return model.eval()


def free():
    gc.collect()
    torch.cuda.empty_cache()


@torch.no_grad()
def predict(model, rows, out_path):
    # greedy decoding; rows hold the system and user turns, the assistant turn is the gold
    prompts = [
        tok.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
        for r in rows
    ]
    order = sorted(range(len(rows)), key=lambda i: len(prompts[i]))
    texts = {}
    for k in range(0, len(order), EVAL_BATCH_SIZE):
        idx = order[k : k + EVAL_BATCH_SIZE]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        for i, g in zip(idx, gen):
            texts[i] = tok.decode(g[enc["input_ids"].shape[1] :], skip_special_tokens=True)
    with open(out_path, "w", encoding="utf-8") as fh:
        for i, r in enumerate(rows):
            fh.write(json.dumps({"id": r["id"], "text": texts[i]}, ensure_ascii=False) + "\n")


def score(preds_path, out_path):
    necrohub(
        "eval-extraction",
        "--data",
        f"{WORK}/datasets",
        "--split",
        "test",
        "--predictions",
        preds_path,
        "--out",
        out_path,
    )
    return json.loads(Path(out_path).read_text("utf-8"))

## Baseline: the base model, no fine-tuning

Same extraction system prompt, greedy decoding, 4-bit load, test split (up to `MAX_EVAL` examples).
The scorer counts a gold record without a prediction as invalid, so with `MAX_EVAL` set both models
are scored over the full split and the cap shows up as missing predictions.

In [ ]:
RESULTS = os.path.abspath("results")
os.makedirs(RESULTS, exist_ok=True)
t = time.time()
model = load_4bit()
predict(model, test_extraction, f"{WORK}/preds_base.jsonl")
del model
free()
WALL["base_predict_s"] = round(time.time() - t, 1)
base_scores = score(f"{WORK}/preds_base.jsonl", f"{RESULTS}/score_base.json")
print(json.dumps(base_scores, indent=1))

## QLoRA training

TRL `SFTTrainer` on the train split (extraction and QA examples), QLoRA nf4 with double quantization,
LoRA on every linear projection, loss on the assistant turns only, validation loss each epoch.
Checkpoints go to `/content/ckpt`; after a Colab disconnect run the setup cells again and this one
resumes from the last checkpoint.

In [ ]:
train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows])
val_ds = Dataset.from_list([{"messages": r["messages"]} for r in val_rows])
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=BNB, dtype=torch.float16, device_map={"": 0}
)
lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
cfg = SFTConfig(
    output_dir=CKPT,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    bf16=False,
    assistant_only_loss=True,
    max_length=MAX_LENGTH,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=20,
    seed=SEED,
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora,
    processing_class=tok,
)
t = time.time()
has_ckpt = any(p.name.startswith("checkpoint-") for p in Path(CKPT).glob("*"))
trainer.train(resume_from_checkpoint=True if has_ckpt else None)
WALL["train_s"] = round(time.time() - t, 1)
ADAPTER = f"{WORK}/adapter"
trainer.save_model(ADAPTER)
history = [h for h in trainer.state.log_history if "eval_loss" in h]
print(history)
del trainer, model
free()

## Fine-tuned predictions and comparison

In [ ]:
t = time.time()
model = load_4bit(ADAPTER)
predict(model, test_extraction, f"{WORK}/preds_tuned.jsonl")
del model
free()
WALL["tuned_predict_s"] = round(time.time() - t, 1)
tuned_scores = score(f"{WORK}/preds_tuned.jsonl", f"{RESULTS}/score_tuned.json")


def pick(s, f):
    return s["field_accuracy"].get(f, s.get(f))


names = [*base_scores["field_accuracy"], "all_fields_exact", "valid_json_rate"]
rows = [(f, pick(base_scores, f), pick(tuned_scores, f)) for f in names]
for k in ("injuries", "organ_findings"):
    rows.append((f"{k} f1", base_scores[k]["f1"], tuned_scores[k]["f1"]))
print(f"{'metric':28}{'base':>10}{'tuned':>10}")
for name, b, f in rows:
    print(f"{name:28}{b:>10}{f:>10}")

## Export: merge, GGUF, Ollama

Merges the adapter into a 16-bit copy of the base model, converts it with llama.cpp at a pinned tag,
quantizes to Q4_K_M and copies the Modelfile from `deploy/ollama/Modelfile`.

In [ ]:
t = time.time()
base16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map={"": 0})
merged = PeftModel.from_pretrained(base16, ADAPTER).merge_and_unload()
MERGED = f"{WORK}/merged"
merged.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
del merged, base16
free()

LLAMA = "/content/llama.cpp"
sh(f"git clone --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp {LLAMA}")
sh(f"pip install -q {LLAMA}/gguf-py sentencepiece protobuf")
sh(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF")
sh(f"cmake --build {LLAMA}/build --target llama-quantize -j 4")
F16 = f"{WORK}/necrohub-extract-f16.gguf"
GGUF = "/content/necrohub-extract-q4_k_m.gguf"
sh(f"python {LLAMA}/convert_hf_to_gguf.py {MERGED} --outtype f16 --outfile {F16}")
sh(f"{LLAMA}/build/bin/llama-quantize {F16} {GGUF} Q4_K_M")
sh("cp necrohub/deploy/ollama/Modelfile /content/Modelfile")
WALL["export_s"] = round(time.time() - t, 1)
print(os.path.getsize(GGUF) / 1e9, "GB")

In [ ]:
if subprocess.run("which ollama", shell=True, capture_output=True, check=False).returncode == 0:
    sh("cd /content && ollama create necrohub-extract -f Modelfile")
    sample = test_extraction[0]["messages"][1]["content"]
    run = subprocess.run(
        ["ollama", "run", "necrohub-extract", sample], capture_output=True, text=True, check=False
    )
    print(run.stdout[:1500])
else:
    print("ollama is not installed here; copy the GGUF and the Modelfile to a machine that has it")

## Results

Writes `results/qlora_extraction.json`. Copy it to `docs/results/` in the repository; the GGUF stays
out of git.

In [ ]:
WALL["total_s"] = round(time.time() - T0, 1)
result = {
    "schema_version": 1,
    "necrohub_ref": NECROHUB_REF,
    "datasets_sha256": DATASET_SHA,
    "base_model": BASE_MODEL,
    "gpu": torch.cuda.get_device_name(0),
    "versions": VERSIONS,
    "hyperparameters": {
        "seed": SEED,
        "n_cases": N_CASES,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "max_length": MAX_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_eval": MAX_EVAL,
        "llama_cpp_tag": LLAMA_CPP_TAG,
    },
    "eval_loss_per_epoch": [h["eval_loss"] for h in history],
    "base": base_scores,
    "fine_tuned": tuned_scores,
    "wall_times_s": WALL,
}
Path(f"{RESULTS}/qlora_extraction.json").write_text(
    json.dumps(result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("copy results/qlora_extraction.json to docs/results/ in the repository")